# as-strided-windowing composite — cx2: Conv2d forward via as_strided patches + einops.einsum

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `as-strided-windowing`, `einops-einsum`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "as-strided-windowing"
DD_ATOM_IDS = ["as-strided-windowing", "einops-einsum"]
DD_SUBTOPICS = ["PyTorch: as_strided windowing", "Einops: Deep Learning"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

ARENA's `conv2d_minimal` is essentially two atoms:

1. **as-strided-windowing** — build a 6-D view of all input patches with shape `(B, C_in, H_out, W_out, K, K)`. The outer four dims index a patch; the inner two index inside one patch.
2. **einops-einsum** — contract that view against the weight tensor `(C_out, C_in, K, K)` along the `C_in, K, K` axes. The result is `(B, C_out, H_out, W_out)` — the conv output.

The einsum string spells out the joint shape:
`einsum(patches, weight, 'b c h w kh kw, o c kh kw -> b o h w')`

Two label notes:
- `c` and `kh/kw` appear on BOTH inputs → contracted (summed over).
- `b`, `h`, `w`, `o` appear on ONE side → preserved as output dims.

The composition: this drill does the FULL forward — strided view, then one einsum call. Reference checks against `t.nn.functional.conv2d`.

### Composite Exercise — Conv2d forward via as_strided patches + einops.einsum

**Atoms exercised together**: `as-strided-windowing`, `einops-einsum`

Implement `cx2_conv2d_via_einsum(x, w)`.

- `x`: float tensor of shape `(B, C_in, H, W)`. Assume contiguous.
- `w`: float tensor of shape `(C_out, C_in, K, K)`. Assume square kernels, stride=1, pad=0.

Return a float tensor of shape `(B, C_out, H_out, W_out)` matching `F.conv2d(x, w, bias=None, stride=1, padding=0)` up to fp32 tolerance.

1. **conv-output-shape** — compute `H_out = H - K + 1`, `W_out = W - K + 1`.
2. **as-strided-windowing** — build a 6-D view of shape `(B, C_in, H_out, W_out, K, K)`. The stride tuple uses `x.stride()` for the `(B, C_in, K, K)` inner motions and the spatial strides again for the `(H_out, W_out)` outer motions.
3. **einops-einsum** — `einops.einsum(patches, w, 'b c h w kh kw, o c kh kw -> b o h w')`.

The test cross-checks against `F.conv2d` on randomized inputs.

In [ ]:
def cx2_conv2d_via_einsum(x, w):
    B, C_in, H, W = x.shape
    C_out, _, KH, KW = w.shape
    # conv-output-shape (stride=1, pad=0).
    H_out = H - KH + 1
    W_out = W - KW + 1
    # Atom A (as-strided-windowing): build (B, C_in, H_out, W_out, KH, KW) view.
    sB, sC, sH, sW = x.stride()
    patches = x.as_strided(
        size=(B, C_in, H_out, W_out, KH, KW),
        stride=(sB, sC, sH, sW, sH, sW),
    )
    # Atom B (einops-einsum): contract c, kh, kw between patches and weight.
    return einops.einsum(patches, w, 'b c h w kh kw, o c kh kw -> b o h w')


<details><summary>Show solution — cx2</summary>

```python
def cx2_conv2d_via_einsum(x, w):
    B, C_in, H, W = x.shape
    C_out, _, KH, KW = w.shape
    # conv-output-shape (stride=1, pad=0).
    H_out = H - KH + 1
    W_out = W - KW + 1
    # Atom A (as-strided-windowing): build (B, C_in, H_out, W_out, KH, KW) view.
    sB, sC, sH, sW = x.stride()
    patches = x.as_strided(
        size=(B, C_in, H_out, W_out, KH, KW),
        stride=(sB, sC, sH, sW, sH, sW),
    )
    # Atom B (einops-einsum): contract c, kh, kw between patches and weight.
    return einops.einsum(patches, w, 'b c h w kh kw, o c kh kw -> b o h w')
```

The stride tuple `(sB, sC, sH, sW, sH, sW)` is the load-bearing piece — the spatial strides appear TWICE because the same 2-D pattern walks both the patch origin (outer (H_out, W_out)) and the inside of one patch (inner (KH, KW)). Once the view exists, einsum does the entire conv arithmetic in a single contraction — no nested loops, no im2col copy.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx2',
        'subtopics': ["PyTorch: as_strided windowing", "Einops: Deep Learning"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()